In [1]:
import os
import mysql.connector
from mysql.connector import Error
from decimal import Decimal

In [3]:
connection = mysql.connector.connect(
    host=os.getenv("MYSQL_HOST", "localhost"),
    port=int(os.getenv("MYSQL_PORT", "3306")),
    user=os.getenv("MYSQL_USER", "root"),
    password=os.getenv("MYSQL_PASSWORD", "A@b@1234"),
    autocommit=False
)
cursor = connection.cursor(dictionary=True)

def execute_sql(statement, params=None):
    cursor.execute(statement, params or ())
    if cursor.with_rows:
        rows = cursor.fetchall()
        for row in rows:
            print(row)
        return rows
    connection.commit()
    print(f"Query OK. Rows affected: {cursor.rowcount}")

def reset_database():
    cursor.execute("CREATE DATABASE IF NOT EXISTS cle_week3")
    cursor.execute("USE cle_week3")
    connection.commit()

reset_database()

In [4]:
cursor.execute("DROP TABLE IF EXISTS sales_lines")
cursor.execute("DROP TABLE IF EXISTS category_budget")

cursor.execute("""
CREATE TABLE sales_lines (
    sale_id INT PRIMARY KEY,
    sale_date DATE NOT NULL,
    category VARCHAR(50) NOT NULL,
    amount DECIMAL(12,2) NOT NULL
)
""")

cursor.execute("""
CREATE TABLE category_budget (
    category VARCHAR(50) PRIMARY KEY,
    Q1 DECIMAL(12,2) NOT NULL,
    Q2 DECIMAL(12,2) NOT NULL,
    Q3 DECIMAL(12,2) NOT NULL,
    Q4 DECIMAL(12,2) NOT NULL
)
""")

sales_data = [
    (1, '2025-01-15', 'Electronics', 5000),
    (2, '2025-02-20', 'Stationery', 800),
    (3, '2025-04-03', 'Electronics', 7000),
    (4, '2025-05-11', 'Home', 2500),
    (5, '2025-06-30', 'Stationery', 600),
    (6, '2025-07-19', 'Electronics', 6500),
    (7, '2025-08-08', 'Home', 3000),
    (8, '2025-10-01', 'Electronics', 9000),
    (9, '2025-11-15', 'Stationery', 1200),
    (10, '2025-12-24', 'Home', 4000),
    (11, '2025-12-26', 'Electronics', 1000)
]

budget_data = [
    ('Electronics', 6000, 6000, 7000, 10000),
    ('Home', 1000, 2000, 3000, 4000),
    ('Stationery', 1000, 1000, 500, 1500)
]

cursor.executemany(
    "INSERT INTO sales_lines VALUES (%s, %s, %s, %s)",
    sales_data
)
cursor.executemany(
    "INSERT INTO category_budget VALUES (%s, %s, %s, %s, %s)",
    budget_data
)
connection.commit()

In [5]:
query_q31_a = """
SELECT
    category,
    SUM(CASE WHEN QUARTER(sale_date) = 1 THEN amount ELSE 0 END) AS Q1,
    SUM(CASE WHEN QUARTER(sale_date) = 2 THEN amount ELSE 0 END) AS Q2,
    SUM(CASE WHEN QUARTER(sale_date) = 3 THEN amount ELSE 0 END) AS Q3,
    SUM(CASE WHEN QUARTER(sale_date) = 4 THEN amount ELSE 0 END) AS Q4,
    SUM(amount) AS total
FROM sales_lines
WHERE sale_date >= '2025-01-01' AND sale_date < '2026-01-01'
GROUP BY category
ORDER BY category
"""
execute_sql(query_q31_a)

{'category': 'Electronics', 'Q1': Decimal('5000.00'), 'Q2': Decimal('7000.00'), 'Q3': Decimal('6500.00'), 'Q4': Decimal('10000.00'), 'total': Decimal('28500.00')}
{'category': 'Home', 'Q1': Decimal('0.00'), 'Q2': Decimal('2500.00'), 'Q3': Decimal('3000.00'), 'Q4': Decimal('4000.00'), 'total': Decimal('9500.00')}
{'category': 'Stationery', 'Q1': Decimal('800.00'), 'Q2': Decimal('600.00'), 'Q3': Decimal('0.00'), 'Q4': Decimal('1200.00'), 'total': Decimal('2600.00')}


[{'category': 'Electronics',
  'Q1': Decimal('5000.00'),
  'Q2': Decimal('7000.00'),
  'Q3': Decimal('6500.00'),
  'Q4': Decimal('10000.00'),
  'total': Decimal('28500.00')},
 {'category': 'Home',
  'Q1': Decimal('0.00'),
  'Q2': Decimal('2500.00'),
  'Q3': Decimal('3000.00'),
  'Q4': Decimal('4000.00'),
  'total': Decimal('9500.00')},
 {'category': 'Stationery',
  'Q1': Decimal('800.00'),
  'Q2': Decimal('600.00'),
  'Q3': Decimal('0.00'),
  'Q4': Decimal('1200.00'),
  'total': Decimal('2600.00')}]

In [6]:
query_q31_b = """
SELECT category, 'Q1' AS quarter, Q1 AS budget FROM category_budget
UNION ALL
SELECT category, 'Q2', Q2 FROM category_budget
UNION ALL
SELECT category, 'Q3', Q3 FROM category_budget
UNION ALL
SELECT category, 'Q4', Q4 FROM category_budget
ORDER BY category, quarter
"""
execute_sql(query_q31_b)

{'category': 'Electronics', 'quarter': 'Q1', 'budget': Decimal('6000.00')}
{'category': 'Electronics', 'quarter': 'Q2', 'budget': Decimal('6000.00')}
{'category': 'Electronics', 'quarter': 'Q3', 'budget': Decimal('7000.00')}
{'category': 'Electronics', 'quarter': 'Q4', 'budget': Decimal('10000.00')}
{'category': 'Home', 'quarter': 'Q1', 'budget': Decimal('1000.00')}
{'category': 'Home', 'quarter': 'Q2', 'budget': Decimal('2000.00')}
{'category': 'Home', 'quarter': 'Q3', 'budget': Decimal('3000.00')}
{'category': 'Home', 'quarter': 'Q4', 'budget': Decimal('4000.00')}
{'category': 'Stationery', 'quarter': 'Q1', 'budget': Decimal('1000.00')}
{'category': 'Stationery', 'quarter': 'Q2', 'budget': Decimal('1000.00')}
{'category': 'Stationery', 'quarter': 'Q3', 'budget': Decimal('500.00')}
{'category': 'Stationery', 'quarter': 'Q4', 'budget': Decimal('1500.00')}


[{'category': 'Electronics', 'quarter': 'Q1', 'budget': Decimal('6000.00')},
 {'category': 'Electronics', 'quarter': 'Q2', 'budget': Decimal('6000.00')},
 {'category': 'Electronics', 'quarter': 'Q3', 'budget': Decimal('7000.00')},
 {'category': 'Electronics', 'quarter': 'Q4', 'budget': Decimal('10000.00')},
 {'category': 'Home', 'quarter': 'Q1', 'budget': Decimal('1000.00')},
 {'category': 'Home', 'quarter': 'Q2', 'budget': Decimal('2000.00')},
 {'category': 'Home', 'quarter': 'Q3', 'budget': Decimal('3000.00')},
 {'category': 'Home', 'quarter': 'Q4', 'budget': Decimal('4000.00')},
 {'category': 'Stationery', 'quarter': 'Q1', 'budget': Decimal('1000.00')},
 {'category': 'Stationery', 'quarter': 'Q2', 'budget': Decimal('1000.00')},
 {'category': 'Stationery', 'quarter': 'Q3', 'budget': Decimal('500.00')},
 {'category': 'Stationery', 'quarter': 'Q4', 'budget': Decimal('1500.00')}]

In [9]:
query_q31_c = """
WITH actuals AS (
    SELECT
        category,
        QUARTER(sale_date) AS quarter_number,
        SUM(amount) AS actual
    FROM sales_lines
    WHERE sale_date >= '2025-01-01' AND sale_date < '2026-01-01'
    GROUP BY category, QUARTER(sale_date)
),
budgets AS (
    SELECT category, 'Q1' AS quarter, Q1 AS budget FROM category_budget
    UNION ALL
    SELECT category, 'Q2', Q2 FROM category_budget
    UNION ALL
    SELECT category, 'Q3', Q3 FROM category_budget
    UNION ALL
    SELECT category, 'Q4', Q4 FROM category_budget
),
all_categories AS (
    SELECT category FROM category_budget
)
SELECT
    c.category,
    b.quarter,
    COALESCE(a.actual, 0) AS actual,
    b.budget,
    COALESCE(a.actual, 0) - b.budget AS variance
FROM all_categories c
JOIN budgets b ON b.category = c.category
LEFT JOIN actuals a
    ON a.category = b.category
   AND CONCAT('Q', a.quarter_number) = b.quarter
ORDER BY c.category, b.quarter
"""
execute_sql(query_q31_c)

{'category': 'Electronics', 'quarter': 'Q1', 'actual': Decimal('5000.00'), 'budget': Decimal('6000.00'), 'variance': Decimal('-1000.00')}
{'category': 'Electronics', 'quarter': 'Q2', 'actual': Decimal('7000.00'), 'budget': Decimal('6000.00'), 'variance': Decimal('1000.00')}
{'category': 'Electronics', 'quarter': 'Q3', 'actual': Decimal('6500.00'), 'budget': Decimal('7000.00'), 'variance': Decimal('-500.00')}
{'category': 'Electronics', 'quarter': 'Q4', 'actual': Decimal('10000.00'), 'budget': Decimal('10000.00'), 'variance': Decimal('0.00')}
{'category': 'Home', 'quarter': 'Q1', 'actual': Decimal('0.00'), 'budget': Decimal('1000.00'), 'variance': Decimal('-1000.00')}
{'category': 'Home', 'quarter': 'Q2', 'actual': Decimal('2500.00'), 'budget': Decimal('2000.00'), 'variance': Decimal('500.00')}
{'category': 'Home', 'quarter': 'Q3', 'actual': Decimal('3000.00'), 'budget': Decimal('3000.00'), 'variance': Decimal('0.00')}
{'category': 'Home', 'quarter': 'Q4', 'actual': Decimal('4000.00'), 

[{'category': 'Electronics',
  'quarter': 'Q1',
  'actual': Decimal('5000.00'),
  'budget': Decimal('6000.00'),
  'variance': Decimal('-1000.00')},
 {'category': 'Electronics',
  'quarter': 'Q2',
  'actual': Decimal('7000.00'),
  'budget': Decimal('6000.00'),
  'variance': Decimal('1000.00')},
 {'category': 'Electronics',
  'quarter': 'Q3',
  'actual': Decimal('6500.00'),
  'budget': Decimal('7000.00'),
  'variance': Decimal('-500.00')},
 {'category': 'Electronics',
  'quarter': 'Q4',
  'actual': Decimal('10000.00'),
  'budget': Decimal('10000.00'),
  'variance': Decimal('0.00')},
 {'category': 'Home',
  'quarter': 'Q1',
  'actual': Decimal('0.00'),
  'budget': Decimal('1000.00'),
  'variance': Decimal('-1000.00')},
 {'category': 'Home',
  'quarter': 'Q2',
  'actual': Decimal('2500.00'),
  'budget': Decimal('2000.00'),
  'variance': Decimal('500.00')},
 {'category': 'Home',
  'quarter': 'Q3',
  'actual': Decimal('3000.00'),
  'budget': Decimal('3000.00'),
  'variance': Decimal('0.00')}

In [10]:
cursor.execute("DROP PROCEDURE IF EXISTS sync_product_master")
cursor.execute("DROP TABLE IF EXISTS product_sync_actions")
cursor.execute("DROP TABLE IF EXISTS product_staging")
cursor.execute("DROP TABLE IF EXISTS product_master")

cursor.execute("""
CREATE TABLE product_master (
    product_id VARCHAR(10) PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    price DECIMAL(10,2) NOT NULL,
    is_active TINYINT(1) NOT NULL DEFAULT 1
)
""")

cursor.execute("""
CREATE TABLE product_staging (
    product_id VARCHAR(10) PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    price DECIMAL(10,2) NOT NULL,
    is_deleted CHAR(1) NOT NULL
)
""")

cursor.execute("""
CREATE TABLE product_sync_actions (
    product_id VARCHAR(10) NOT NULL,
    action VARCHAR(10) NOT NULL
)
""")

cursor.executemany(
    "INSERT INTO product_master VALUES (%s, %s, %s, %s)",
    [
        ('P1', 'Wireless Mouse', 799, 1),
        ('P2', 'USB-C Hub', 1999, 1),
        ('P3', 'Notebook A5', 120, 1),
        ('P4', 'Gel Pen Pack', 250, 1)
    ]
)

cursor.executemany(
    "INSERT INTO product_staging VALUES (%s, %s, %s, %s)",
    [
        ('P2', 'USB-C Hub 7-in-1', 2199, 'N'),
        ('P3', 'Notebook A5', 120, 'N'),
        ('P4', 'Gel Pen Pack', 250, 'Y'),
        ('P5', 'Desk Lamp', 1499, 'N'),
        ('P6', 'Water Bottle', 499, 'N')
    ]
)
connection.commit()

In [11]:
sync_product_master_sql = """
CREATE PROCEDURE sync_product_master()
BEGIN
    DELETE FROM product_sync_actions;

    INSERT INTO product_sync_actions (product_id, action)
    SELECT
        s.product_id,
        CASE
            WHEN s.is_deleted = 'Y' THEN 'DELETE'
            WHEN p.product_id IS NULL THEN 'INSERT'
            ELSE 'UPDATE'
        END
    FROM product_staging s
    LEFT JOIN product_master p ON p.product_id = s.product_id
    WHERE
        (s.is_deleted = 'Y' AND p.product_id IS NOT NULL)
        OR
        (s.is_deleted = 'N' AND p.product_id IS NULL)
        OR
        (s.is_deleted = 'N' AND p.product_id IS NOT NULL
         AND (p.product_name <> s.product_name OR p.price <> s.price));

    UPDATE product_master p
    JOIN product_staging s ON s.product_id = p.product_id
    SET p.product_name = s.product_name,
        p.price = s.price
    WHERE s.is_deleted = 'N'
      AND (p.product_name <> s.product_name OR p.price <> s.price);

    INSERT INTO product_master (product_id, product_name, price, is_active)
    SELECT s.product_id, s.product_name, s.price, 1
    FROM product_staging s
    LEFT JOIN product_master p ON p.product_id = s.product_id
    WHERE s.is_deleted = 'N' AND p.product_id IS NULL;

    DELETE p
    FROM product_master p
    JOIN product_staging s ON s.product_id = p.product_id
    WHERE s.is_deleted = 'Y';
END
"""
cursor.execute(sync_product_master_sql)
connection.commit()
cursor.callproc("sync_product_master")
connection.commit()
execute_sql("SELECT * FROM product_master ORDER BY product_id")
execute_sql("SELECT * FROM product_sync_actions ORDER BY product_id")

{'product_id': 'P1', 'product_name': 'Wireless Mouse', 'price': Decimal('799.00'), 'is_active': 1}
{'product_id': 'P2', 'product_name': 'USB-C Hub 7-in-1', 'price': Decimal('2199.00'), 'is_active': 1}
{'product_id': 'P3', 'product_name': 'Notebook A5', 'price': Decimal('120.00'), 'is_active': 1}
{'product_id': 'P5', 'product_name': 'Desk Lamp', 'price': Decimal('1499.00'), 'is_active': 1}
{'product_id': 'P6', 'product_name': 'Water Bottle', 'price': Decimal('499.00'), 'is_active': 1}
{'product_id': 'P2', 'action': 'UPDATE'}
{'product_id': 'P4', 'action': 'DELETE'}
{'product_id': 'P5', 'action': 'INSERT'}
{'product_id': 'P6', 'action': 'INSERT'}


[{'product_id': 'P2', 'action': 'UPDATE'},
 {'product_id': 'P4', 'action': 'DELETE'},
 {'product_id': 'P5', 'action': 'INSERT'},
 {'product_id': 'P6', 'action': 'INSERT'}]

In [12]:
cursor.callproc("sync_product_master")
connection.commit()
execute_sql("SELECT * FROM product_master ORDER BY product_id")
execute_sql("SELECT * FROM product_sync_actions ORDER BY product_id")

{'product_id': 'P1', 'product_name': 'Wireless Mouse', 'price': Decimal('799.00'), 'is_active': 1}
{'product_id': 'P2', 'product_name': 'USB-C Hub 7-in-1', 'price': Decimal('2199.00'), 'is_active': 1}
{'product_id': 'P3', 'product_name': 'Notebook A5', 'price': Decimal('120.00'), 'is_active': 1}
{'product_id': 'P5', 'product_name': 'Desk Lamp', 'price': Decimal('1499.00'), 'is_active': 1}
{'product_id': 'P6', 'product_name': 'Water Bottle', 'price': Decimal('499.00'), 'is_active': 1}


[]

In [13]:
cursor.execute("DROP PROCEDURE IF EXISTS apply_customer_updates")
cursor.execute("DROP TABLE IF EXISTS customer_updates")
cursor.execute("DROP TABLE IF EXISTS CustomerDim")

cursor.execute("""
CREATE TABLE CustomerDim (
    CustomerID INT NOT NULL,
    CustomerName VARCHAR(100) NOT NULL,
    Address VARCHAR(150) NOT NULL,
    EffectiveStartDate DATE NOT NULL,
    EffectiveEndDate DATE NOT NULL,
    IsCurrent TINYINT(1) NOT NULL,
    PRIMARY KEY (CustomerID, EffectiveStartDate)
)
""")

cursor.execute("""
CREATE TABLE customer_updates (
    CustomerID INT PRIMARY KEY,
    CustomerName VARCHAR(100) NOT NULL,
    Address VARCHAR(150) NOT NULL
)
""")

cursor.executemany(
    "INSERT INTO CustomerDim VALUES (%s, %s, %s, %s, %s, %s)",
    [
        (1, 'John Doe', '123 Main St', '2023-01-01', '9999-12-31', 1),
        (2, 'Alice Johnson', '456 Elm St', '2023-01-01', '9999-12-31', 1),
        (3, 'Bob Smith', '789 Oak St', '2023-01-01', '9999-12-31', 1)
    ]
)

cursor.executemany(
    "INSERT INTO customer_updates VALUES (%s, %s, %s)",
    [
        (1, 'John Doe', 'Ajmer'),
        (4, 'David Richard', 'Mumbai'),
        (3, 'Bob Smith', 'Chennai'),
        (5, 'Eva Dsouza', 'Mumbai')
    ]
)
connection.commit()

In [14]:
apply_customer_updates_sql = """
CREATE PROCEDURE apply_customer_updates(IN p_effective_date DATE)
BEGIN
    UPDATE CustomerDim d
    JOIN customer_updates u ON u.CustomerID = d.CustomerID
    SET d.EffectiveEndDate = DATE_SUB(p_effective_date, INTERVAL 1 DAY),
        d.IsCurrent = 0
    WHERE d.IsCurrent = 1
      AND (d.CustomerName <> u.CustomerName OR d.Address <> u.Address);

    INSERT INTO CustomerDim (
        CustomerID, CustomerName, Address,
        EffectiveStartDate, EffectiveEndDate, IsCurrent
    )
    SELECT
        u.CustomerID, u.CustomerName, u.Address,
        p_effective_date, '9999-12-31', 1
    FROM customer_updates u
    WHERE NOT EXISTS (
        SELECT 1
        FROM CustomerDim d
        WHERE d.CustomerID = u.CustomerID
          AND d.IsCurrent = 1
          AND d.CustomerName = u.CustomerName
          AND d.Address = u.Address
    );
END
"""
cursor.execute(apply_customer_updates_sql)
connection.commit()
cursor.callproc("apply_customer_updates", ['2026-10-01'])
connection.commit()
execute_sql("SELECT * FROM CustomerDim ORDER BY CustomerID, EffectiveStartDate")

{'CustomerID': 1, 'CustomerName': 'John Doe', 'Address': '123 Main St', 'EffectiveStartDate': datetime.date(2023, 1, 1), 'EffectiveEndDate': datetime.date(2026, 9, 30), 'IsCurrent': 0}
{'CustomerID': 1, 'CustomerName': 'John Doe', 'Address': 'Ajmer', 'EffectiveStartDate': datetime.date(2026, 10, 1), 'EffectiveEndDate': datetime.date(9999, 12, 31), 'IsCurrent': 1}
{'CustomerID': 2, 'CustomerName': 'Alice Johnson', 'Address': '456 Elm St', 'EffectiveStartDate': datetime.date(2023, 1, 1), 'EffectiveEndDate': datetime.date(9999, 12, 31), 'IsCurrent': 1}
{'CustomerID': 3, 'CustomerName': 'Bob Smith', 'Address': '789 Oak St', 'EffectiveStartDate': datetime.date(2023, 1, 1), 'EffectiveEndDate': datetime.date(2026, 9, 30), 'IsCurrent': 0}
{'CustomerID': 3, 'CustomerName': 'Bob Smith', 'Address': 'Chennai', 'EffectiveStartDate': datetime.date(2026, 10, 1), 'EffectiveEndDate': datetime.date(9999, 12, 31), 'IsCurrent': 1}
{'CustomerID': 4, 'CustomerName': 'David Richard', 'Address': 'Mumbai', 'Ef

[{'CustomerID': 1,
  'CustomerName': 'John Doe',
  'Address': '123 Main St',
  'EffectiveStartDate': datetime.date(2023, 1, 1),
  'EffectiveEndDate': datetime.date(2026, 9, 30),
  'IsCurrent': 0},
 {'CustomerID': 1,
  'CustomerName': 'John Doe',
  'Address': 'Ajmer',
  'EffectiveStartDate': datetime.date(2026, 10, 1),
  'EffectiveEndDate': datetime.date(9999, 12, 31),
  'IsCurrent': 1},
 {'CustomerID': 2,
  'CustomerName': 'Alice Johnson',
  'Address': '456 Elm St',
  'EffectiveStartDate': datetime.date(2023, 1, 1),
  'EffectiveEndDate': datetime.date(9999, 12, 31),
  'IsCurrent': 1},
 {'CustomerID': 3,
  'CustomerName': 'Bob Smith',
  'Address': '789 Oak St',
  'EffectiveStartDate': datetime.date(2023, 1, 1),
  'EffectiveEndDate': datetime.date(2026, 9, 30),
  'IsCurrent': 0},
 {'CustomerID': 3,
  'CustomerName': 'Bob Smith',
  'Address': 'Chennai',
  'EffectiveStartDate': datetime.date(2026, 10, 1),
  'EffectiveEndDate': datetime.date(9999, 12, 31),
  'IsCurrent': 1},
 {'CustomerID':

In [15]:
cursor.execute("DROP TABLE IF EXISTS sales_big")
cursor.execute("""
CREATE TABLE sales_big (
    order_id INT PRIMARY KEY,
    customer_id VARCHAR(6),
    order_date DATE,
    amount DECIMAL(10,2)
)
""")
connection.commit()

ProgrammingError: 1054 (42S22): Unknown column 'd5.n' in 'field list'

In [17]:
populate_sales_big = """
INSERT INTO sales_big (order_id, customer_id, order_date, amount)
SELECT
    n,
    CONCAT('C', LPAD(MOD(n - 1, 5000), 4, '0')),
    DATE_ADD('2023-01-01', INTERVAL MOD(n - 1, 1095) DAY),
    MOD(n * 37, 5000) + 99
FROM (
    SELECT
        d0.n + d1.n * 10 + d2.n * 100
        + d3.n * 1000 + d4.n * 10000 + 1 AS n
    FROM
        (SELECT 0 n UNION ALL SELECT 1 UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4 UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL SELECT 8 UNION ALL SELECT 9) d0
    CROSS JOIN
        (SELECT 0 n UNION ALL SELECT 1 UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4 UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL SELECT 8 UNION ALL SELECT 9) d1
    CROSS JOIN
        (SELECT 0 n UNION ALL SELECT 1 UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4 UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL SELECT 8 UNION ALL SELECT 9) d2
    CROSS JOIN
        (SELECT 0 n UNION ALL SELECT 1 UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4 UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL SELECT 8 UNION ALL SELECT 9) d3
    CROSS JOIN
        (SELECT 0 n UNION ALL SELECT 1 UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4 UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL SELECT 8 UNION ALL SELECT 9) d4
) numbers
WHERE n <= 500000
"""

In [18]:
cursor.execute(populate_sales_big)
connection.commit()

In [19]:
execute_sql("SELECT COUNT(*) AS total_rows FROM sales_big")

{'total_rows': 100000}


[{'total_rows': 100000}]

In [22]:

cursor.execute("DROP TABLE IF EXISTS performance_results")

cursor.execute("""
CREATE TABLE performance_results (
    run VARCHAR(10) PRIMARY KEY,
    main_operator VARCHAR(20),
    rows_read BIGINT
)
""")
connection.commit()

customer_lookup_query = """
SELECT order_id, order_date, amount
FROM sales_big
WHERE customer_id = 'C0042'
  AND order_date >= '2025-03-01'
  AND order_date < '2025-04-01'
"""

cursor.execute("EXPLAIN " + customer_lookup_query)
before_plan = cursor.fetchall()

cursor.execute("""
CREATE INDEX idx_sales_big_customer_date
ON sales_big (customer_id, order_date)
""")
connection.commit()

cursor.execute("EXPLAIN " + customer_lookup_query)
after_plan = cursor.fetchall()


def save_plan_result(run_name, plan):
    plan_row = next(
        (row for row in plan if row.get("table") == "sales_big"),
        plan[0]
    )

    access_type = plan_row.get("type", "")
    rows_read = plan_row.get("rows", 0)
    main_operator = "Scan" if access_type == "ALL" else "Seek"

    cursor.execute(
        """
        INSERT INTO performance_results
        (run, main_operator, rows_read)
        VALUES (%s, %s, %s)
        """,
        (run_name, main_operator, rows_read)
    )
    connection.commit()


save_plan_result("Before", before_plan)
save_plan_result("After", after_plan)

print("Before Index:")
for row in before_plan:
    print(row)

print("\nAfter Index:")
for row in after_plan:
    print(row)

print("\nPerformance Results:")
execute_sql("""
SELECT * FROM performance_results
ORDER BY CASE run
    WHEN 'Before' THEN 1
    WHEN 'After' THEN 2
END
""")


Before Index:
{'id': 1, 'select_type': 'SIMPLE', 'table': 'sales_big', 'partitions': None, 'type': 'ALL', 'possible_keys': None, 'key': None, 'key_len': None, 'ref': None, 'rows': 71969, 'filtered': 1.11, 'Extra': 'Using where'}

After Index:
{'id': 1, 'select_type': 'SIMPLE', 'table': 'sales_big', 'partitions': None, 'type': 'range', 'possible_keys': 'idx_sales_big_customer_date', 'key': 'idx_sales_big_customer_date', 'key_len': '31', 'ref': None, 'rows': 1, 'filtered': 100.0, 'Extra': 'Using index condition'}

Performance Results:
{'run': 'Before', 'main_operator': 'Scan', 'rows_read': 71969}
{'run': 'After', 'main_operator': 'Seek', 'rows_read': 1}


[{'run': 'Before', 'main_operator': 'Scan', 'rows_read': 71969},
 {'run': 'After', 'main_operator': 'Seek', 'rows_read': 1}]

In [23]:
raw_name = ' wireless MOUSE             '
raw_sku = 'sku-0017'
raw_price = '₹1,299.50'
raw_qty = ' 3 '
gst_rate = 18

name = ' '.join(raw_name.split()).title()
sku = raw_sku.strip().upper()
price = Decimal(raw_price.replace('₹', '').replace(',', '').strip())
qty = int(raw_qty.strip())
subtotal = (price * qty).quantize(Decimal('0.01'))
gst = (subtotal * Decimal(gst_rate) / Decimal('100')).quantize(Decimal('0.01'))
total = (subtotal + gst).quantize(Decimal('0.01'))

print({
    'name': name,
    'sku': sku,
    'price': float(price),
    'qty': qty,
    'subtotal': float(subtotal),
    'gst': float(gst),
    'total': float(total)
})

print(
    f"{name} | {sku} | {qty} x {price:,.2f} = {subtotal:,.2f} | "
    f"GST {gst:,.2f} | Total {total:,.2f}"
)

{'name': 'Wireless Mouse', 'sku': 'SKU-0017', 'price': 1299.5, 'qty': 3, 'subtotal': 3898.5, 'gst': 701.73, 'total': 4600.23}
Wireless Mouse | SKU-0017 | 3 x 1,299.50 = 3,898.50 | GST 701.73 | Total 4,600.23


In [24]:
cart = {'P1': 2, 'P2': 1, 'P3': 5}

cart['P5'] = 1
cart['P2'] += 3
removed_quantity = cart.pop('P3')
quantity_p9 = cart.get('P9', 0)
product_ids = sorted(cart.keys())
different_products = len(cart)
total_units = sum(cart.values())
highest_quantity_product = max(cart, key=cart.get)

print(cart)
print(removed_quantity)
print(quantity_p9)
print(product_ids)
print(f"{different_products} products, {total_units} units")
print(highest_quantity_product)

{'P1': 2, 'P2': 4, 'P5': 1}
5
0
['P1', 'P2', 'P5']
3 products, 7 units
P2


In [25]:
crm_ids = ['C01', 'C02', 'C03', 'C02', 'C04', 'C07']
billing_ids = ['C02', 'C04', 'C05', 'C06', 'C04']

crm_unique = list(dict.fromkeys(crm_ids))
common_ids = sorted(set(crm_ids) & set(billing_ids))
crm_only = sorted(set(crm_ids) - set(billing_ids))
billing_only = sorted(set(billing_ids) - set(crm_ids))
all_unique_count = len(set(crm_ids) | set(billing_ids))

print("A:", crm_unique)
print("B:", common_ids)
print("C:", crm_only)
print("D:", billing_only)
print("E:", all_unique_count)

A: ['C01', 'C02', 'C03', 'C04', 'C07']
B: ['C02', 'C04']
C: ['C01', 'C03', 'C07']
D: ['C05', 'C06']
E: 7


In [26]:
config = {
    'pipeline': 'daily_sales',
    'source': {
        'type': 'csv',
        'path': '/data/in/sales.csv',
        'delimiter': ','
    },
    'target': {
        'db': ('db.internal', 5432),
        'table': 'stage.sales'
    },
    'columns': ['order_id', 'store', 'amount'],
    'active': True
}

source_path = config['source']['path']
host, port = config['target']['db']
config['target']['table'] = 'stage.sales_v2'
config['columns'].append('sale_date')
amount_exists = 'amount' in config['columns']
discount_exists = 'discount' in config['columns']
config['retries'] = 3
top_level_keys = list(config.keys())

print(source_path)
print(f"host = '{host}', port = {port}")
print(config['target']['table'])
print(config['columns'])
print(f"amount: {amount_exists}, discount: {discount_exists}")
print(config['retries'])
print(top_level_keys)

/data/in/sales.csv
host = 'db.internal', port = 5432
stage.sales_v2
['order_id', 'store', 'amount', 'sale_date']
amount: True, discount: False
3
['pipeline', 'source', 'target', 'columns', 'active', 'retries']


In [27]:
cursor.close()
connection.close()